# IPL Match Winner Prediction – Model Training
Trains a `RandomForestClassifier` on pre-match features only and exports `model.pkl` and `encoder.pkl` for the Flask app.

## 1. Imports

In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42

## 2. Data Loading & Cleaning

In [2]:
df = pd.read_csv("ipl.csv")
print("Raw shape:", df.shape)
df.head()

Raw shape: (1169, 19)


,team1,team2,match_date,toss_winner,toss_decision,winner,player_of_match,venue,city,team1_players,team2_players,season,match_number,match_type,result,result_margin,target_runs,target_overs,super_over
0,Royal Challengers Bangalore,Kolkata Knight Riders,2008-04-18,Royal Challengers Bangalore,field,Kolkata Knight Riders,BB McCullum,M Chinnaswamy Stadium,Bangalore,"R Dravid, W Jaffer, V Kohli, JH Kallis, CL Whi...","SC Ganguly, BB McCullum, RT Ponting, DJ Hussey...",1,1,League,runs,140.0,223.0,20.0,N
1,Kings XI Punjab,Chennai Super Kings,2008-04-19,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Chandigarh,"K Goel, JR Hopes, KC Sangakkara, Yuvraj Singh,...","PA Patel, ML Hayden, MEK Hussey, MS Dhoni, SK ...",1,2,League,runs,33.0,241.0,20.0,N
2,Delhi Daredevils,Rajasthan Royals,2008-04-19,Rajasthan Royals,bat,Delhi Daredevils,MF Maharoof,Feroz Shah Kotla,Delhi,"G Gambhir, V Sehwag, S Dhawan, MK Tiwary, KD K...","T Kohli, YK Pathan, SR Watson, M Kaif, DS Lehm...",1,3,League,wickets,9.0,130.0,20.0,N
3,Mumbai Indians,Royal Challengers Bangalore,2008-04-20,Mumbai Indians,bat,Royal Challengers Bangalore,MV Boucher,Wankhede Stadium,Mumbai,"L Ronchi, ST Jayasuriya, DJ Thornely, RV Uthap...","S Chanderpaul, R Dravid, LRPL Taylor, JH Kalli...",1,4,League,wickets,5.0,166.0,20.0,N
4,Kolkata Knight Riders,Deccan Chargers,2008-04-20,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Eden Gardens,Kolkata,"WP Saha, BB McCullum, RT Ponting, SC Ganguly, ...","AC Gilchrist, Y Venugopal Rao, VVS Laxman, A S...",1,5,League,wickets,5.0,111.0,20.0,N


Drop rows with no winner (abandoned / no-result matches).

In [3]:
df = df.dropna(subset=["winner"]).reset_index(drop=True)
print("After dropping null winners:", df.shape)

After dropping null winners: (1146, 19)


Impute missing `city` from `venue`. First use the most common city recorded for the same venue; venues that never have a city (Dubai / Sharjah stadiums) fall back to the city named in the venue string.

In [4]:
print("Missing city before:", df["city"].isnull().sum())

venue_to_city = (
    df.dropna(subset=["city"])
      .groupby("venue")["city"]
      .agg(lambda s: s.mode().iloc[0])
      .to_dict()
)
df["city"] = df["city"].fillna(df["venue"].map(venue_to_city))

# Venues with no city anywhere in the data: infer from the venue name
fallback = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
}
df["city"] = df["city"].fillna(df["venue"].map(fallback))

print("Missing city after:", df["city"].isnull().sum())

Missing city before: 48
Missing city after: 0


Drop data-leakage columns (they are only known after the match) and the player-list columns for a clean baseline.

In [5]:
leakage_cols = ["player_of_match", "result", "result_margin", "target_runs", "target_overs", "super_over"]
player_cols = ["team1_players", "team2_players"]
df = df.drop(columns=leakage_cols + player_cols)
df.columns.tolist()

['team1',
 'team2',
 'match_date',
 'toss_winner',
 'toss_decision',
 'winner',
 'venue',
 'city',
 'season',
 'match_number',
 'match_type']

### Standardise names
Franchises were renamed over the years and some stadiums appear under several spellings. Merging them stops the model (and the web dropdowns) from treating the same entity as different ones.

In [6]:
team_aliases = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

venue_aliases = {
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Eden Gardens, Kolkata": "Eden Gardens",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium",
    "Sardar Patel Stadium, Motera": "Narendra Modi Stadium, Ahmedabad",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",
    "Zayed Cricket Stadium, Abu Dhabi": "Sheikh Zayed Stadium",
}

for col in ["team1", "team2", "toss_winner", "winner"]:
    df[col] = df[col].replace(team_aliases)
df["venue"] = df["venue"].replace(venue_aliases)

print("Teams :", pd.concat([df.team1, df.team2]).nunique())
print("Venues:", df.venue.nunique())

Teams : 15
Venues: 38


## 3. Feature Engineering & Encoding
Keep only pre-match features. One shared `LabelEncoder` is used for `team1`, `team2`, `toss_winner` **and** the target `winner`, so a given team always maps to the same integer in every column.

In [7]:
feature_cols = ["team1", "team2", "venue", "toss_winner", "toss_decision"]
target_col = "winner"

data = df[feature_cols + [target_col]].copy()
data.isnull().sum()

team1            0
team2            0
venue            0
toss_winner      0
toss_decision    0
winner           0
dtype: int64

In [8]:
# Shared team encoder – fitted on the union of every column that contains a team name
all_teams = pd.concat([data["team1"], data["team2"], data["toss_winner"], data["winner"]]).unique()
team_encoder = LabelEncoder().fit(sorted(all_teams))

venue_encoder = LabelEncoder().fit(sorted(data["venue"].unique()))
toss_decision_encoder = LabelEncoder().fit(sorted(data["toss_decision"].unique()))

X = pd.DataFrame({
    "team1": team_encoder.transform(data["team1"]),
    "team2": team_encoder.transform(data["team2"]),
    "venue": venue_encoder.transform(data["venue"]),
    "toss_winner": team_encoder.transform(data["toss_winner"]),
    "toss_decision": toss_decision_encoder.transform(data["toss_decision"]),
})
y = team_encoder.transform(data[target_col])

print("Teams:", team_encoder.classes_.tolist())
print("Toss decisions:", toss_decision_encoder.classes_.tolist())
X.head()

Teams: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
Toss decisions: ['bat', 'field']


,team1,team2,venue,toss_winner,toss_decision
0,13,6,16,13,1
1,10,0,26,0,0
2,2,11,0,11,0
3,8,13,37,8,0
4,6,1,10,1,0


## 4. Model Training (80/20 split)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)

Train: (916, 5) Test: (230, 5)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

## 5. Evaluation

In [10]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")

labels = np.unique(np.concatenate([y_test, y_pred]))
print(classification_report(
    y_test, y_pred,
    labels=labels,
    target_names=team_encoder.inverse_transform(labels),
    zero_division=0,
))

Accuracy: 0.4957

                             precision    recall  f1-score   support

        Chennai Super Kings       0.53      0.71      0.61        28
            Deccan Chargers       1.00      0.17      0.29         6
             Delhi Capitals       0.46      0.46      0.46        28
              Gujarat Lions       0.50      0.33      0.40         3
             Gujarat Titans       0.00      0.00      0.00         5
       Kochi Tuskers Kerala       0.00      0.00      0.00         2
      Kolkata Knight Riders       0.52      0.44      0.48        25
       Lucknow Super Giants       0.00      0.00      0.00         6
             Mumbai Indians       0.33      0.58      0.42        24
              Pune Warriors       0.00      0.00      0.00         5
               Punjab Kings       0.35      0.35      0.35        20
           Rajasthan Royals       0.58      0.48      0.52        23
     Rising Pune Supergiant       0.00      0.00      0.00         2
Royal Challenge

**Reading the result:** the model predicts the winner out of every franchise, so some of its raw predictions can be a team that is not even playing. The Flask app therefore compares the model's probabilities for only `team1` and `team2` and picks the higher one. The check below measures accuracy under that same rule.

In [11]:
proba = model.predict_proba(X_test)
class_index = {c: i for i, c in enumerate(model.classes_)}

def pick_between(row_proba, t1, t2):
    p1 = row_proba[class_index[t1]] if t1 in class_index else 0.0
    p2 = row_proba[class_index[t2]] if t2 in class_index else 0.0
    return t1 if p1 >= p2 else t2

restricted = [
    pick_between(p, t1, t2)
    for p, t1, t2 in zip(proba, X_test["team1"], X_test["team2"])
]
print(f"Accuracy (choosing between the two playing teams): {accuracy_score(y_test, restricted):.4f}")

Accuracy (choosing between the two playing teams): 0.5217


## 6. Export `model.pkl` and `encoder.pkl`

In [12]:
encoders = {
    "team_encoder": team_encoder,
    "venue_encoder": venue_encoder,
    "toss_decision_encoder": toss_decision_encoder,
    "feature_cols": feature_cols,
    "team_aliases": team_aliases,
    "venue_aliases": venue_aliases,
}

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("encoder.pkl", "wb") as f:
    pickle.dump(encoders, f)

print("Saved model.pkl and encoder.pkl")

Saved model.pkl and encoder.pkl


In [13]:
# Sanity check: reload and predict one example
with open("model.pkl", "rb") as f:
    m = pickle.load(f)
with open("encoder.pkl", "rb") as f:
    enc = pickle.load(f)

sample = pd.DataFrame([{
    "team1": enc["team_encoder"].transform(["Mumbai Indians"])[0],
    "team2": enc["team_encoder"].transform(["Chennai Super Kings"])[0],
    "venue": enc["venue_encoder"].transform(["Wankhede Stadium"])[0],
    "toss_winner": enc["team_encoder"].transform(["Chennai Super Kings"])[0],
    "toss_decision": enc["toss_decision_encoder"].transform(["field"])[0],
}])
print(enc["team_encoder"].inverse_transform(m.predict(sample)))

['Chennai Super Kings']
